# Laude dashboard

In [ ]:
## Load the study data

In [ ]:
import base64
import json
import os
from datetime import date, datetime
from zoneinfo import ZoneInfo

import pandas as pd
from jupyterhealth_client import JupyterHealthClient

try:
    from provider_app import (
        jhe_auth,
        launch_context,
    )

    ctx = launch_context.current()
except Exception as e:
    print("No SMART context", e)
    client = JupyterHealthClient()
    PATIENT_ID = 40032
    patient = client.get_patient(PATIENT_ID)

else:
    client = jhe_auth.client_for_launch(ctx)
    patient = client.lookup_patient(
        external_id=ctx.patient_mrn,
        system=os.getenv("MRN_IDENTIFIER_SYSTEM"),
    )
    PATIENT_ID = patient["id"]


me = client.get_user()
# Alex K.'s records in the Laude Demo study.
STUDY_ID = 30014
FHIR_SOURCE_ID = 90009
STUDY_TIMEZONE = ZoneInfo("America/Los_Angeles")
EXCHANGE_URL = os.environ["JHE_URL"]

In [ ]:
from IPython.display import Markdown, display

user = client.get_user()
study = client.get_study(STUDY_ID)


def md(md_text: str):
    display(Markdown(md_text))


md(f"""

## SMART Launch credentials

**Practitioner**: [{user["id"]}] {user["firstName"]} {user["lastName"]} 

**Patient**: [{patient["id"]}] {patient["nameGiven"]} {patient["nameFamily"]}

**Study**: [{study["id"]}] {study["name"]} (Organization: {study["organization"]["name"]})

- {study["description"]}
""")

In [ ]:
# The eight mapped-Observation families this dashboard reads, as Open mHealth / IEEE codes.
OMH = "https://w3id.org/openmhealth"
IEEE = "https://w3id.org/ieee1752"
CODE_BLOOD_GLUCOSE = "omh:blood-glucose:4.0"
CODE_STEP_COUNT = "omh:step-count:3.0"
CODE_HEART_RATE = "omh:heart-rate:2.0"
CODE_FOOD_ENTRY = "ieee:food-entry:0.1"
CODE_SLEEP_EPISODE = "ieee:sleep-episode:1.0"
CODE_BODY_WEIGHT = "omh:body-weight:3.0"
CODE_BODY_HEIGHT = "omh:body-height:2.0"
CODE_BLOOD_PRESSURE = "omh:blood-pressure:4.0"
FAMILY_SYSTEMS = {
    CODE_BLOOD_GLUCOSE: OMH,
    CODE_STEP_COUNT: OMH,
    CODE_HEART_RATE: OMH,
    CODE_FOOD_ENTRY: IEEE,
    CODE_SLEEP_EPISODE: IEEE,
    CODE_BODY_WEIGHT: OMH,
    CODE_BODY_HEIGHT: OMH,
    CODE_BLOOD_PRESSURE: OMH,
}
families = {
    code: list(
        client.list_observations(
            patient_id=PATIENT_ID, code=f"{system}|{code}", limit=None
        )
    )
    for code, system in FAMILY_SYSTEMS.items()
}

# The auxiliary clinical context (labs, BMI, HRV) through FHIR resource search.
source_url = f"{EXCHANGE_URL}/FHIR/R5/Observation"
fhir_source = f"https://jupyterhealth.org/fhir/fhir-source/{FHIR_SOURCE_ID}"
auxiliary_observations = []
while source_url:
    bundle = client.session.get(
        source_url, params={"_source": fhir_source, "_count": 1000}, timeout=60
    ).json()
    auxiliary_observations.extend(
        entry["resource"] for entry in bundle.get("entry", []) if "resource" in entry
    )
    source_url = next(
        (
            link["url"]
            for link in bundle.get("link", [])
            if link.get("relation") == "next"
        ),
        None,
    )

In [ ]:
## Decode raw records into data frames

In [ ]:
def decode_attachments(resources):
    decoded = []
    for resource in resources:
        payload = json.loads(base64.b64decode(resource["valueAttachment"]["data"]))
        decoded.append(
            {"id": resource["id"], "header": payload["header"], "body": payload["body"]}
        )
    return decoded


def local_time(text):
    value = datetime.fromisoformat(str(text))
    return value if value.tzinfo else value.replace(tzinfo=STUDY_TIMEZONE)


def source_name(item):
    provenance = item["header"].get("acquisition_provenance") or {}
    return str(provenance.get("source_name") or "Exchange")


def frame(rows, time_column="recorded_at"):
    data = pd.DataFrame(rows)
    if not data.empty:
        data[time_column] = pd.to_datetime(data[time_column])
        data = data.sort_values(time_column).reset_index(drop=True)
    return data


def cgm_frame():
    rows = []
    for item in decode_attachments(families[CODE_BLOOD_GLUCOSE]):
        body = item["body"]
        rows.append(
            {
                "recorded_at": local_time(body["effective_time_frame"]["date_time"]),
                "mg_dl": float(body["blood_glucose"]["value"]),
                "source": source_name(item),
            }
        )
    return frame(rows)


def steps_frame():
    rows = []
    for item in decode_attachments(families[CODE_STEP_COUNT]):
        body = item["body"]
        interval = body["effective_time_frame"]["time_interval"]
        rows.append(
            {
                "recorded_at": local_time(interval["start_date_time"]),
                "steps": int(body["step_count"]["value"]),
                "source": source_name(item),
            }
        )
    return frame(rows)


def meals_frame():
    rows = []
    for item in decode_attachments(families[CODE_FOOD_ENTRY]):
        body = item["body"]
        rows.append(
            {
                "meal_id": item["id"],
                "occurred_at": local_time(body["effective_time_frame"]["date_time"]),
                "label": str(body.get("food_name") or "Meal"),
                "meal_type": str(body.get("meal") or "meal"),
                "carbohydrate_g": float(
                    (body.get("carbohydrate") or {}).get("value") or 0
                ),
                "protein_g": float((body.get("protein") or {}).get("value") or 0),
                "fat_g": float((body.get("fat") or {}).get("value") or 0),
                "fiber_g": float((body.get("fiber") or {}).get("value") or 0),
                "calories_kcal": float((body.get("calories") or {}).get("value") or 0),
                "source": source_name(item),
            }
        )
    data = frame(rows, "occurred_at")
    data["recorded_at"] = data["occurred_at"]
    return data


def sleep_frame():
    rows = []
    for item in decode_attachments(families[CODE_SLEEP_EPISODE]):
        body = item["body"]
        interval = body["effective_time_frame"]["time_interval"]
        rows.append(
            {
                "night_id": item["id"],
                "start_at": local_time(interval["start_date_time"]),
                "end_at": local_time(interval["end_date_time"]),
                "recorded_at": local_time(interval["end_date_time"]),
                "total_sleep_minutes": float(
                    (body.get("total_sleep_time") or {}).get("value") or 0
                ),
                "efficiency_percent": float(
                    (body.get("sleep_efficiency_percentage") or {}).get("value") or 0
                ),
                "deep_minutes": float(
                    (body.get("deep_sleep_duration") or {}).get("value") or 0
                ),
                "light_minutes": float(
                    (body.get("light_sleep_duration") or {}).get("value") or 0
                ),
                "rem_minutes": float(
                    (body.get("rem_sleep_duration") or {}).get("value") or 0
                ),
                "awakenings": int(body.get("number_of_awakenings") or 0),
                "source": source_name(item),
            }
        )
    return frame(rows)


def heart_rate_frame():
    rows = []
    for item in decode_attachments(families[CODE_HEART_RATE]):
        body = item["body"]
        rows.append(
            {
                "recorded_at": local_time(body["effective_time_frame"]["date_time"]),
                "bpm": float(body["heart_rate"]["value"]),
                "source": source_name(item),
            }
        )
    return frame(rows)


def body_frame(code, key):
    rows = []
    for item in decode_attachments(families[code]):
        body = item["body"]
        rows.append(
            {
                "recorded_at": local_time(body["effective_time_frame"]["date_time"]),
                "value": float(body[key]["value"]),
                "unit": str(body[key].get("unit") or ""),
                "source": source_name(item),
            }
        )
    return frame(rows)


def blood_pressure_frame():
    rows = []
    for item in decode_attachments(families[CODE_BLOOD_PRESSURE]):
        body = item["body"]
        rows.append(
            {
                "recorded_at": local_time(body["effective_time_frame"]["date_time"]),
                "systolic_mm_hg": float(body["systolic_blood_pressure"]["value"]),
                "diastolic_mm_hg": float(body["diastolic_blood_pressure"]["value"]),
                "source": source_name(item),
            }
        )
    return frame(rows)


def auxiliary_frame():
    rows = []
    for resource in auxiliary_observations:
        coding = (resource.get("code") or {}).get("coding") or [{}]
        quantity = resource.get("valueQuantity") or {}
        when = resource.get("effectiveDateTime")
        if quantity.get("value") is None or when is None:
            continue
        rows.append(
            {
                "code": coding[0].get("code"),
                "recorded_at": local_time(when),
                "value": float(quantity["value"]),
                "unit": str(quantity.get("unit") or ""),
            }
        )
    return frame(rows)


cgm = cgm_frame()
steps = steps_frame()
meals = meals_frame()
sleep = sleep_frame()
heart_rate = heart_rate_frame()
weights = body_frame(CODE_BODY_WEIGHT, "body_weight")
heights = body_frame(CODE_BODY_HEIGHT, "body_height")
blood_pressure = blood_pressure_frame()
auxiliary = auxiliary_frame()

In [ ]:
## Prepare the baseline and follow-up sessions

In [ ]:
SESSION_WINDOWS = (
    {
        "name": "baseline",
        "start": date(2026, 8, 23),
        "end": date(2026, 9, 5),
        "label": "Baseline · Aug 23–Sep 5, 2026",
    },
    {
        "name": "follow_up",
        "start": date(2026, 9, 13),
        "end": date(2026, 9, 26),
        "label": "Follow-up · Sep 13–26, 2026",
    },
)


def session_of(moment):
    day = moment.date()
    for window in SESSION_WINDOWS:
        if window["start"] <= day <= window["end"]:
            return window["name"]
    return None


for active in (cgm, steps, meals, sleep, heart_rate):
    active["session"] = active["recorded_at"].map(session_of)

In [ ]:
## Calculate the report metrics

In [ ]:
BAND_KEYS = ("very_low", "low", "target", "high", "very_high")
VERY_LOW_BOUND = 54.0
LOW_BOUND = 70.0
TARGET_HIGH = 180.0
HIGH_BOUND = 250.0
GMI_INTERCEPT = 3.31
GMI_SLOPE = 0.02392


def mean_glucose(values):
    return round(float(values.mean()), 1) if len(values) else None


def gmi_percent(values):
    return (
        round(GMI_INTERCEPT + GMI_SLOPE * float(values.mean()), 1)
        if len(values)
        else None
    )


def coefficient_of_variation(values):
    if len(values) < 2:
        return None
    return round(float(values.std(ddof=1) / values.mean() * 100), 1)


def time_active(recording):
    stamps = sorted(set(recording["recorded_at"]))
    if not stamps:
        return None
    if len(stamps) == 1:
        return 100.0
    expected = int((stamps[-1] - stamps[0]).total_seconds() // 300) + 1
    return round(min(100.0, len(stamps) / expected * 100), 1)


def time_in_ranges(values):
    counts = {key: 0 for key in BAND_KEYS}
    total = len(values)
    for value in values:
        if value < VERY_LOW_BOUND:
            counts["very_low"] += 1
        elif value < LOW_BOUND:
            counts["low"] += 1
        elif value <= TARGET_HIGH:
            counts["target"] += 1
        elif value <= HIGH_BOUND:
            counts["high"] += 1
        else:
            counts["very_high"] += 1
    return {
        key: (round(counts[key] / total * 100, 1) if total else 0.0)
        for key in BAND_KEYS
    }


def percentile_profile(recording, step_minutes=15):
    buckets = {}
    for moment, value in zip(recording["recorded_at"], recording["mg_dl"]):
        bucket = (moment.hour * 60 + moment.minute) // step_minutes * step_minutes
        buckets.setdefault(bucket, []).append(float(value))
    profile = []
    for minute in sorted(buckets):
        series = pd.Series(sorted(buckets[minute]))
        profile.append(
            {
                "minute": minute,
                "p05": round(float(series.quantile(0.05)), 1),
                "p25": round(float(series.quantile(0.25)), 1),
                "p50": round(float(series.quantile(0.50)), 1),
                "p75": round(float(series.quantile(0.75)), 1),
                "p95": round(float(series.quantile(0.95)), 1),
            }
        )
    return profile


agp_by_session = {}
for window in SESSION_WINDOWS:
    recording = cgm[cgm["session"] == window["name"]]
    values = recording["mg_dl"]
    agp_by_session[window["name"]] = {
        "metrics": {
            "mean_mg_dl": mean_glucose(values),
            "gmi_percent": gmi_percent(values),
            "cv_percent": coefficient_of_variation(values),
        },
        "time_active_percent": time_active(recording),
        "bands": time_in_ranges(values),
        "percentiles": percentile_profile(recording),
    }

In [ ]:
## Compare the previous and current measurements

In [ ]:
def aux_points(code):
    rows = auxiliary[auxiliary["code"] == code]
    return list(zip(rows["recorded_at"], rows["value"]))


def weight_points():
    return list(zip(weights["recorded_at"], weights["value"]))


def height_points():
    return list(zip(heights["recorded_at"], heights["value"]))


def bmi_points():
    return aux_points("body-mass-index")


def when(moment):
    return moment.strftime("%b %-d, %Y") if moment is not None else "date not recorded"


def number(value, decimals):
    return "n/a" if value is None else f"{value:.{decimals}f}"


def signed(delta, decimals):
    return f"{delta:+.{decimals}f}".replace("-", "−")


def direction_of(delta, decimals):
    if abs(delta) < 10 ** -(decimals + 1):
        return "flat", "→"
    return ("down", "↓") if delta < 0 else ("up", "↑")


def comparison_card(label, points, unit, decimals=1):
    ordered = sorted(points, key=lambda item: item[0])
    if not ordered:
        return None
    current_moment, current_value = ordered[-1]
    card = {
        "label": label,
        "current": f"{number(current_value, decimals)} {unit}".strip(),
        "current_date": when(current_moment),
        "comparison": False,
        "detail": f"One measurement · {when(current_moment)}",
    }
    if len(ordered) < 2:
        return card
    previous_moment, previous_value = ordered[-2]
    delta = current_value - previous_value
    direction, arrow = direction_of(delta, decimals)
    card.update(
        {
            "comparison": True,
            "previous": f"{number(previous_value, decimals)} {unit}".strip(),
            "previous_date": when(previous_moment),
            "delta": f"{signed(delta, decimals)} {unit}".strip(),
            "direction": direction,
            "arrow": arrow,
        }
    )
    return card


def blood_pressure_card():
    ordered = sorted(
        zip(
            blood_pressure["recorded_at"],
            blood_pressure["systolic_mm_hg"],
            blood_pressure["diastolic_mm_hg"],
        ),
        key=lambda item: item[0],
    )
    if not ordered:
        return None
    moment, systolic, diastolic = ordered[-1]
    card = {
        "label": "Blood pressure",
        "current": f"{systolic:.0f}/{diastolic:.0f} mmHg",
        "current_date": when(moment),
        "comparison": False,
        "detail": f"One measurement · {when(moment)}",
    }
    if len(ordered) < 2:
        return card
    previous_moment = ordered[-2][0]
    delta_systolic = systolic - ordered[-2][1]
    delta_diastolic = diastolic - ordered[-2][2]
    direction, arrow = direction_of(delta_systolic, 0)
    card.update(
        {
            "comparison": True,
            "previous": f"{ordered[-2][1]:.0f}/{ordered[-2][2]:.0f} mmHg",
            "previous_date": when(previous_moment),
            "delta": f"{signed(delta_systolic, 0)}/{signed(delta_diastolic, 0)} mmHg",
            "direction": direction,
            "arrow": arrow,
        }
    )
    return card


context_cards = [
    comparison_card("HbA1c", aux_points("hba1c"), "%", 1),
    comparison_card("Fasting glucose", aux_points("fasting_glucose"), "mg/dL", 0),
    comparison_card("HRV (RMSSD)", aux_points("heart-rate-variability-rmssd"), "ms", 0),
    comparison_card("BMI", bmi_points(), "kg/m²", 1),
    blood_pressure_card(),
    comparison_card("Weight", weight_points(), "kg", 1),
    comparison_card("Height", height_points(), "cm", 1),
]

In [ ]:
## Prepare the explorer and the dashboard view

In [ ]:
ACTIVITY_BUCKET_MINUTES = 30


def activity_buckets(session):
    rows = steps[steps["session"] == session]
    if rows.empty:
        return pd.DataFrame(columns=["bucket_at", "steps"])
    return (
        rows.assign(
            bucket_at=rows["recorded_at"].dt.floor(f"{ACTIVITY_BUCKET_MINUTES}min")
        )
        .groupby("bucket_at", as_index=False)["steps"]
        .sum()
        .sort_values("bucket_at")
        .reset_index(drop=True)
    )


def explorer_for(window):
    name = window["name"]
    return {
        "name": name,
        "label": window["label"],
        "glucose": cgm.loc[
            cgm["session"] == name, ["recorded_at", "mg_dl"]
        ].reset_index(drop=True),
        "meals": meals.loc[
            meals["session"] == name,
            ["occurred_at", "label", "meal_type", "carbohydrate_g", "calories_kcal"],
        ].reset_index(drop=True),
        "activity": activity_buckets(name),
        "sleep": sleep.loc[
            sleep["session"] == name,
            [
                "start_at",
                "end_at",
                "deep_minutes",
                "light_minutes",
                "rem_minutes",
                "efficiency_percent",
                "total_sleep_minutes",
            ],
        ].reset_index(drop=True),
    }


explorers = [explorer_for(window) for window in SESSION_WINDOWS]


def initials(name):
    return "".join(part[0] for part in name.split() if part)[:2].upper()


birth_date = datetime.fromisoformat(str(patient["birthDate"])).date()
age_years = (SESSION_WINDOWS[-1]["end"] - birth_date).days // 365
date_span = (
    f"{SESSION_WINDOWS[0]['start']:%b %-d} – {SESSION_WINDOWS[-1]['end']:%b %-d, %Y}"
)

dashboard_view = {
    "patient": {
        "name": "Alex K.",
        "initials": initials("Alex K."),
        "facts": [
            f"{age_years} years · male",
            f"MRN {patient['id']} · {date_span}",
        ],
    },
    "context_cards": [card for card in context_cards if card],
    "sessions": [
        {
            "name": window["name"],
            "label": window["label"],
            "period": f"14 days · {window['start']:%b %-d} – {window['end']:%b %-d, %Y}",
            "time_active_percent": agp_by_session[window["name"]][
                "time_active_percent"
            ],
            "metrics": agp_by_session[window["name"]]["metrics"],
            "bands": [
                {"key": key, "percent": percent}
                for key, percent in agp_by_session[window["name"]]["bands"].items()
            ],
            "percentiles": agp_by_session[window["name"]]["percentiles"],
        }
        for window in SESSION_WINDOWS
    ],
    "explorers": explorers,
    "provenance": {
        "study": f"{study['name']} ({study['id']})",
        "fhir_source_id": FHIR_SOURCE_ID,
    },
}

In [ ]:
## Inspect the results

In [ ]:
from IPython.display import display as show

VOILA = os.environ.get("VOILA_REQUEST_URL")
if VOILA:
    # skip these in voila
    def show(*args):
        pass


show(
    pd.DataFrame(
        [
            {
                "session": session["label"],
                "mean_mg_dl": session["metrics"]["mean_mg_dl"],
                "gmi_percent": session["metrics"]["gmi_percent"],
                "cv_percent": session["metrics"]["cv_percent"],
                "time_active_percent": session["time_active_percent"],
            }
            for session in dashboard_view["sessions"]
        ]
    )
)
show(meals[["session", "occurred_at", "meal_type", "carbohydrate_g", "calories_kcal"]])
show(auxiliary)

In [ ]:
## Build the dashboard

In [ ]:
from jupyterhealth_dashboard.ui import build_dashboard

dashboard = build_dashboard(dashboard_view)
display(dashboard)